# World-model probe

Evaluation only: the real maze and the exact DP are used to score the model, nothing here feeds training.

**1. Start values.** The NOR value head read at prefix 0 (only the start cell visible) at every start cell, turned into an expected reward with the bin representatives, against the exact random walk's expected reward from the DP with the same representatives (so binning cancels). Reported as the signed error against distance from the goal, and as a log ratio because far-start rewards are tiny.

**2. Imagined transitions.** Thousands of rollouts from random recorded prefixes with the model's own actions, dynamics head and END. Every imagined transition is checked against the real maze: given the cell and the chosen action, was the predicted next cell the one the walls allow?

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}
RUN = "late_threshold/mc_all_s0"  #@param {type:"string"}
MAZE_KW = dict(binning="geometric", n_bins=24)   # must match the run (its K is checked)
COND = "threshold"                                # must match the run: "bin" or "threshold" (stored in its params.pkl)


In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
import os, pickle, numpy as np, jax.numpy as jnp
import matplotlib.pyplot as plt
from maze_consistency.dataset import load as load_data
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import ModelConfig, MazeTransformer, make_forward
from maze_consistency.dp import compute_ground_truth
from maze_consistency.train import load_run
from maze_consistency import probe as P
from maze_consistency.evaluate import make_state_logits, make_cell_logits

import subprocess, sys
if not os.path.exists("data/canonical/rollouts.npz"):          # the dataset is built, not stored in the repo
    subprocess.run([sys.executable, "run.py", "dataset"], check=True)
MAZE, DATA = load_data(**MAZE_KW)
TOK = Tokenizer(MAZE, cond=COND)
GT = compute_ground_truth(MAZE)
params, cfg = load_run(RUN)
import pickle as _pk
_meta = _pk.load(open(os.path.join(os.environ.get('RUNS_DIR', 'runs'), RUN, 'params.pkl'), 'rb'))
if 'cond' in _meta and _meta['cond'] != COND:
    raise ValueError(f"run was trained with cond={_meta['cond']!r}; set COND to match")
assert cfg.K == MAZE.K, f"run has K={cfg.K} but MAZE_KW gives K={MAZE.K}; set MAZE_KW to the run's binning"
MODEL = MazeTransformer(cfg); FWD = make_forward(MODEL, TOK)
print(RUN, "|", cfg, "|", MAZE)
LIVE = [k for k in range(MAZE.K) if k not in MAZE.empty_bins]
_best_at = lambda dd: int(MAZE.best_bin(np.flatnonzero(MAZE.dist == dd)[0]))
FAR_BINS = sorted({_best_at(20), _best_at(15), _best_at(10)})
print("usable bins:", LIVE, "| far starts' best bins:", FAR_BINS)
STATE_LOGITS, CELL_LOGITS = make_state_logits(MODEL, TOK), make_cell_logits(MODEL, TOK)


## 1. Start values: model vs exact random walk

In [ ]:
SV = P.start_values(FWD, params, TOK, MAZE, GT)
print(f"{len(SV['cell'])} start cells | mean signed err {SV['err'].mean():+.4f} | mean |log ratio| {np.abs(SV['log_ratio']).mean():.3f}")
print(f"\n{'dist':>5s}{'n':>4s}{'model R':>10s}{'exact R':>10s}{'exact(unbinned)':>17s}{'err':>10s}{'log ratio':>11s}")
for dd in np.unique(SV['dist']):
    m = SV['dist'] == dd
    print(f"{dd:5d}{m.sum():4d}{SV['model_R'][m].mean():10.4f}{SV['exact_R_binned'][m].mean():10.4f}{SV['exact_R'][m].mean():17.4f}"
          f"{SV['err'][m].mean():+10.4f}{SV['log_ratio'][m].mean():+11.3f}")


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(SV['dist'], SV['err'], s=18, alpha=.7); ax[0].axhline(0, color='k', lw=.8)
ax[0].set_xlabel('distance from goal'); ax[0].set_ylabel('model E[R] - exact E[R]  (signed)')
ax[1].scatter(SV['dist'], SV['log_ratio'], s=18, alpha=.7); ax[1].axhline(0, color='k', lw=.8)
ax[1].set_xlabel('distance from goal'); ax[1].set_ylabel('log(model E[R] / exact E[R])')
o = np.argsort(SV['dist'])
ax[2].scatter(SV['dist'], SV['model_R'], s=18, alpha=.7, label='model')
ax[2].plot(SV['dist'][o], SV['exact_R_binned'][o], 'k.', ms=4, label='exact (binned)')
ax[2].set_yscale('log'); ax[2].set_xlabel('distance from goal'); ax[2].set_ylabel('E[R] under the random policy'); ax[2].legend()
for a in ax: a.grid(alpha=.3)
fig.suptitle(f'{RUN}: NOR value head at prefix 0'); fig.tight_layout(); plt.show()


## 2. Imagined transitions: how often does the world model break the maze?

`wrong_dir`: the predicted cell is the current cell or an open neighbour, but not where the action leads (moved when it should have bumped, bumped when it should have moved, or went sideways). `wall`: the predicted cell is a wall. `teleport`: an open cell that is neither the current cell nor a neighbour.

In [ ]:
N_ROLLOUTS, MAX_STEPS = 2000, 10
MODES = (None, *sorted(set(FAR_BINS) | {LIVE[-1]}))
IT = P.imagined_transitions(params, STATE_LOGITS, CELL_LOGITS, TOK, MAZE, DATA, n=N_ROLLOUTS, seed=0, max_steps=MAX_STEPS, modes=MODES)
name = lambda m: 'NOR' if m < 0 else f'bin {m}'
R = IT['rows']
print(f"{len(IT['ok'])} imagined steps from {N_ROLLOUTS} prefixes per mode, up to {MAX_STEPS} steps each\n")
print(f"{'mode':8s}{'steps':>8s}{'impossible':>12s}{'wrong_dir':>11s}{'wall':>8s}{'teleport':>10s}{'rows w/ any':>13s}{'mean steps':>12s}{'ended':>8s}")
for m in sorted(set(IT['mode'])):
    s, r = IT['mode'] == m, R['mode'] == m
    kc = np.bincount(IT['kind'][s], minlength=4) / s.sum()
    print(f"{name(m):8s}{s.sum():8d}{1 - kc[0]:12.3%}{kc[1]:11.3%}{kc[2]:8.3%}{kc[3]:10.3%}{R['any_bad'][r].mean():13.3f}{R['n_steps'][r].mean():12.2f}{R['ended'][r].mean():8.3f}")


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
depths = np.arange(1, MAX_STEPS + 1)
tau_edges = [0, 1, 5, 20, 50, 100, MAZE.T]
for m in sorted(set(IT['mode'])):
    s = IT['mode'] == m
    ax[0].plot(depths, [1 - IT['ok'][s & (IT['depth'] == k)].mean() if (s & (IT['depth'] == k)).any() else np.nan for k in depths], marker='.', label=name(m))
    tb = [1 - IT['ok'][s & (IT['tau'] >= lo) & (IT['tau'] < hi)].mean() if (s & (IT['tau'] >= lo) & (IT['tau'] < hi)).any() else np.nan for lo, hi in zip(tau_edges[:-1], tau_edges[1:])]
    ax[1].plot(range(len(tb)), tb, marker='.', label=name(m))
ax[0].set_xlabel('imagined step (depth from the prefix)'); ax[0].set_ylabel('fraction of impossible transitions'); ax[0].legend(fontsize=7)
ax[1].set_xticks(range(len(tau_edges) - 1)); ax[1].set_xticklabels([f'{lo}-{hi - 1}' for lo, hi in zip(tau_edges[:-1], tau_edges[1:])])
ax[1].set_xlabel('recorded prefix length'); ax[1].set_ylabel('fraction of impossible transitions')
nor = IT['mode'] == -1
kc = np.bincount(IT['kind'][nor], minlength=4) / nor.sum()
ax[2].bar(IT['kind_names'][1:], kc[1:]); ax[2].set_ylabel('fraction of NOR imagined steps'); ax[2].set_title('impossible transitions by kind (NOR)')
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); plt.show()


In [ ]:
# where do impossible transitions happen? per-cell rate of the NOR world model, drawn on the maze
cnt = np.bincount(IT['cell'][nor], minlength=MAZE.n_cells).astype(float)
bad = np.bincount(IT['cell'][nor], weights=~IT['ok'][nor], minlength=MAZE.n_cells)
rate = np.where(cnt > 0, bad / np.maximum(cnt, 1), np.nan).reshape(MAZE.H, MAZE.W)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(rate, cmap='Reds', vmin=0, vmax=1)
ax.imshow(np.where(MAZE.grid == 0, 0.0, np.nan), cmap='gray', vmin=0, vmax=1)   # walls in black
gy, gx = divmod(int(MAZE.goal), MAZE.W); ax.plot(gx, gy, 'g*', ms=12)
plt.colorbar(im, ax=ax, label='impossible-transition rate (NOR)'); ax.set_title('by current cell; star = goal'); plt.show()
